In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [9]:
import os
path = "/content/drive/MyDrive/BATTERY_SOC_PROJECT"
files = os.listdir("/content/drive/MyDrive/BATTERY_SOC_PROJECT/Dataset_Li-ion")
print(files)

['utils.py', 'Technical Information and Experimental Test Results for LG 18650HG2.pdf', 'Readme file - Description of Experimental Tests.txt', 'n10degC', '25degC', 'n20degC', '0degC', '10degC', '40degC']


In [10]:
import os
import random
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
print(" REPRODUCIBLE PREPROCESSING")
print("Random seed:", RANDOM_SEED)

 REPRODUCIBLE PREPROCESSING
Random seed: 42


In [11]:
path = ("/content/drive/MyDrive/BATTERY_SOC_PROJECT/Dataset_Li-ion")
csv_files = []
for root, dirs, files in os.walk(path):
    for file in files:
        if file.lower().endswith(".csv"):
            csv_files.append(os.path.join(root, file))
csv_files = sorted(csv_files)
print("CSV FILE SEARCH")
print("Dataset path:", path)
print("Total CSV files found:", len(csv_files))
if len(csv_files) == 0:
    raise FileNotFoundError("No CSV files were found. ""Please check the Google Drive dataset path.")


CSV FILE SEARCH
Dataset path: /content/drive/MyDrive/BATTERY_SOC_PROJECT/Dataset_Li-ion
Total CSV files found: 208


In [12]:
def find_header_row(file_path):
    try:
        with open(
            file_path,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as f:
            lines = f.readlines()
        possible_words = ["time","voltage","current","temperature","capacity"]

        for i, line in enumerate(lines[:100]):
            line_lower = line.lower()
            score = sum(
                word in line_lower
                for word in possible_words
            )
            if score >= 2:
                return i
    except Exception:
        pass
    return None

In [13]:
def clean_battery_file(file_path):
    header_row = find_header_row(file_path)
    if header_row is None:
        return None
    try:
        df = pd.read_csv(
            file_path,
            skiprows=header_row,
            low_memory=False,
            on_bad_lines="skip"
        )
        df = df.dropna(axis=1,how="all")
        df.columns = [str(col).strip().replace("\ufeff", "")for col in df.columns]
        df = df.dropna(axis=0, how="all")
        df = df.loc[:,~df.columns.duplicated()]
        df["Source_File"] = os.path.basename(file_path)
        return df
    except Exception:
        return None

In [14]:
cleaned_data = []
failed_files = []
total_files = len(csv_files)
print("PROCESSING BATTERY CSV FILES")
print(f"Total CSV files to process: {total_files}")
print()
for file_number, file_path in enumerate(csv_files,start=1):
    file_name = os.path.basename(file_path)
    df_temp = clean_battery_file(file_path)
    if df_temp is not None and len(df_temp) > 0:
        cleaned_data.append(df_temp)
        print(
            f"{file_number}/{total_files}  "
            f"Processed: {file_name}"
        )
    else:
        failed_files.append(file_name)
        print(
            f"{file_number}/{total_files}  "
            f"Failed: {file_name}"
        )

if len(cleaned_data) == 0:
    raise ValueError("No measurement tables could be extracted ""from the CSV files.")
battery_df = pd.concat(cleaned_data,ignore_index=True,sort=False)


successful_files = len(cleaned_data)
failed_count = len(failed_files)
print("CSV PROCESSING SUMMARY")
print(f"Total CSV files found       : {total_files}")
print( f"Successfully processed      : {successful_files}")
print(f"Failed to process           : {failed_count}")
print(f"Total rows in combined data : {len(battery_df)}")
print(f"Total columns               : {len(battery_df.columns)}")
print(f"Combined dataset shape      : {battery_df.shape}")

if failed_count > 0:
    print("\nFailed files:")
    for file_name in failed_files:
        print(" -", file_name)
else:
    print(
        "\n ALL CSV FILES WERE PROCESSED SUCCESSFULLY!")
print("\nFinal dataset columns:")
for col in battery_df.columns:
    print(" -", col)


PROCESSING BATTERY CSV FILES
Total CSV files to process: 208

1/208  Processed: 585_C20DisCh.csv
2/208  Processed: 585_Dis_0p5C.csv
3/208  Processed: 585_Dis_2C.csv
4/208  Processed: 585_HPPC.csv
5/208  Processed: 589_Cap_1C.csv
6/208  Processed: 589_Charge1.csv
7/208  Processed: 589_Charge2.csv
8/208  Processed: 589_Charge3.csv
9/208  Processed: 589_Charge4.csv
10/208  Processed: 589_Charge5.csv
11/208  Processed: 589_Charge6.csv
12/208  Processed: 589_Charge7.csv
13/208  Processed: 589_Charge8.csv
14/208  Processed: 589_HWFET.csv
15/208  Processed: 589_LA92.csv
16/208  Processed: 589_Mixed1.csv
17/208  Processed: 589_Mixed2.csv
18/208  Processed: 589_UDDS.csv
19/208  Processed: 589_US06.csv
20/208  Processed: 590_Charge10.csv
21/208  Processed: 590_Charge11.csv
22/208  Processed: 590_Charge12.csv
23/208  Processed: 590_Charge13.csv
24/208  Processed: 590_Charge14.csv
25/208  Processed: 590_Charge15.csv
26/208  Processed: 590_Charge16.csv
27/208  Processed: 590_Mixed4.csv
28/208  Proc

In [15]:
print("Current columns in battery_df:")
print(battery_df.columns.tolist())
required_columns = ["Voltage","Current","Temperature","Capacity","Cycle","Status","Source_File"]
missing_columns = [
    col
    for col in required_columns
    if col not in battery_df.columns
]
if len(missing_columns) > 0:
    print("\nMissing required columns:")
    for col in missing_columns:
        print("-", col)
    raise ValueError(
        "Required columns are missing. "
        "Please check the dataset." )
print("\nAll required columns are available.")

Current columns in battery_df:
['Time Stamp', 'Step', 'Status', 'Prog Time', 'Step Time', 'Cycle', 'Cycle Level', 'Procedure', 'Voltage', 'Current', 'Temperature', 'Capacity', 'WhAccu', 'Cnt', 'Source_File']

All required columns are available.


In [16]:
numeric_columns = ["Voltage", "Current","Temperature","Capacity","Cycle"]
if "WhAccu" in battery_df.columns:
    numeric_columns.append("WhAccu")
if "Cnt" in battery_df.columns:
    numeric_columns.append("Cnt")
for col in numeric_columns:
    battery_df[col] = pd.to_numeric(battery_df[col],errors="coerce")
print("\nNumeric conversion completed.")
battery_df = battery_df.dropna(axis=0,how="all")
battery_df = battery_df.dropna(axis=1, how="all")
print("Shape after removing completely empty ""rows/columns:",battery_df.shape)
duplicate_count = battery_df.duplicated().sum()
print("\nDuplicate rows found:",duplicate_count)
battery_df = battery_df.drop_duplicates()
battery_df = battery_df.reset_index(drop=True)
print("Rows after duplicate removal:",len(battery_df))
invalid_voltage = (battery_df["Voltage"] <= 0)
invalid_temperature = ( ~battery_df["Temperature"].between(-50,100))
battery_df.loc[invalid_voltage,"Voltage"] = np.nan
battery_df.loc[invalid_temperature,"Temperature"] = np.nan
battery_df.loc[battery_df["Capacity"] < 0,"Capacity"] = np.nan
if "WhAccu" in battery_df.columns:
    battery_df.loc[battery_df["WhAccu"] < 0,"WhAccu"] = np.nan
print("\nInvalid measurement handling completed.")
print("\nCurrent Dataset Shape:",battery_df)


Numeric conversion completed.
Shape after removing completely empty rows/columns: (4955125, 15)

Duplicate rows found: 5876
Rows after duplicate removal: 4949249

Invalid measurement handling completed.

Current Dataset Shape:                      Time Stamp  Step Status     Prog Time     Step Time  \
0                           NaN   NaN    NaN           NaN           NaN   
1         11/27/2018 8:41:18 PM  22.0    DCH  25:19:08.386  00:01:00.004   
2         11/27/2018 8:42:18 PM  22.0    DCH  25:20:08.381  00:01:59.999   
3         11/27/2018 8:43:18 PM  22.0    DCH  25:21:08.383  00:03:00.001   
4         11/27/2018 8:44:18 PM  22.0    DCH  25:22:08.381  00:03:59.999   
...                         ...   ...    ...           ...           ...   
4949244  12/26/2018 10:37:50 PM  80.0    PAU  32:30:55.745  00:09:59.702   
4949245  12/26/2018 10:37:50 PM  80.0    PAU  32:30:55.846  00:09:59.803   
4949246  12/26/2018 10:37:50 PM  80.0    PAU  32:30:55.945  00:09:59.902   
4949247  12/

In [17]:
print("Status values:")
print(battery_df["Status"].value_counts(dropna=False))
print("\nTime Stamp sample:")
print(battery_df["Time Stamp"].head(10))
print("\nStep sample:")
print(battery_df["Step"].head(10))

Status values:
Status
TABLE    4145008
PAU       721030
CHA        44650
DCH        38334
NaN          208
STO           19
Name: count, dtype: int64

Time Stamp sample:
0                      NaN
1    11/27/2018 8:41:18 PM
2    11/27/2018 8:42:18 PM
3    11/27/2018 8:43:18 PM
4    11/27/2018 8:44:18 PM
5    11/27/2018 8:45:18 PM
6    11/27/2018 8:46:18 PM
7    11/27/2018 8:47:18 PM
8    11/27/2018 8:48:18 PM
9    11/27/2018 8:49:18 PM
Name: Time Stamp, dtype: object

Step sample:
0     NaN
1    22.0
2    22.0
3    22.0
4    22.0
5    22.0
6    22.0
7    22.0
8    22.0
9    22.0
Name: Step, dtype: float64


In [18]:
battery_df["_status_change"] = (battery_df["Status"].astype(str).ne(battery_df["Status"].astype(str).shift()))
battery_df["_file_change"] = (battery_df["Source_File"].astype(str).ne( battery_df["Source_File"].astype(str).shift()))
battery_df["_new_session"] = (battery_df["_status_change"]|battery_df["_file_change"])
battery_df["Charging_Session"] = (battery_df["_new_session"].cumsum())
print("\nNumber of charging sessions:",battery_df["Charging_Session"].nunique())


Number of charging sessions: 1931


In [19]:
charging_mask = (battery_df["Status"].astype(str).str.strip().str.upper().eq("CHA"))
charging_df = battery_df.loc[charging_mask].copy()
print("\nCharging records:",len(charging_df))
if len(charging_df) == 0:
    raise ValueError("No charging records were found.")



Charging records: 44650


In [20]:
charging_df["Final_Charging_Capacity"] = (charging_df.groupby("Charging_Session")["Capacity"].transform("max"))
valid_capacity = (charging_df["Final_Charging_Capacity"] > 0)
charging_df["SOC"] = np.nan
charging_df.loc[valid_capacity,"SOC"] = ( charging_df.loc[valid_capacity,"Capacity"]/charging_df.loc[valid_capacity,"Final_Charging_Capacity"]* 100)
charging_df["SOC"] = (charging_df["SOC"].clip(0, 100))
print("\nSOC target generated.")
print("SOC missing values:",charging_df["SOC"].isna().sum())
print("SOC range:",
    round(charging_df["SOC"].min(),2),
      "to",
    round(charging_df["SOC"].max(), 2))
charging_df["Time Stamp"] = pd.to_datetime(charging_df["Time Stamp"],errors="coerce")

charging_df = charging_df.dropna(subset=["Time Stamp"]).copy()

charging_df["Final_Charging_Time"] = (charging_df.groupby("Charging_Session")["Time Stamp"].transform("max"))

charging_df["Remaining Charging Time"] = (charging_df["Final_Charging_Time"] -charging_df["Time Stamp"]).dt.total_seconds()

charging_df["Remaining Charging Time"] = (charging_df["Remaining Charging Time"].clip(lower=0))

print("\nRemaining charging time created successfully.")
print("Remaining Charging Time range:",
    round(charging_df["Remaining Charging Time"].min(), 2),
    "to",
    round(charging_df["Remaining Charging Time"].max(), 2),
    "seconds")
print(
    "Missing Remaining Charging Time:",
    charging_df["Remaining Charging Time"].isna().sum())


SOC target generated.
SOC missing values: 30502
SOC range: 0.0 to 100.0

Remaining charging time created successfully.
Remaining Charging Time range: 0.0 to 72123.0 seconds
Missing Remaining Charging Time: 0


In [21]:
charging_df["Elapsed Time"] = (charging_df["Time Stamp"] -charging_df.groupby("Charging_Session")["Time Stamp"].transform("min")).dt.total_seconds()


soc_input_variables = ["Voltage","Current","Temperature","Elapsed Time","Cycle"]
if "WhAccu" in charging_df.columns:
    soc_input_variables.append("WhAccu")

time_input_variables = ["Voltage","Current","Temperature","Elapsed Time","Cycle","Capacity"]
if "WhAccu" in charging_df.columns:
    time_input_variables.append("WhAccu")

target_variables = ["SOC","Remaining Charging Time"]
print("\nSOC input variables:")
print(soc_input_variables)
print("\nRemaining-time input variables:")
print(time_input_variables)
print("\nTarget variables:")
print(target_variables)


SOC input variables:
['Voltage', 'Current', 'Temperature', 'Elapsed Time', 'Cycle', 'WhAccu']

Remaining-time input variables:
['Voltage', 'Current', 'Temperature', 'Elapsed Time', 'Cycle', 'Capacity', 'WhAccu']

Target variables:
['SOC', 'Remaining Charging Time']


In [22]:
soc_model_df = charging_df[soc_input_variables +["SOC", "Source_File", "Charging_Session"]].copy()

time_model_df = charging_df[time_input_variables +["Remaining Charging Time", "Source_File", "Charging_Session"]].copy()


soc_model_df = (soc_model_df.dropna(subset=["SOC"]).reset_index(drop=True))

time_model_df = (time_model_df.dropna(subset=["Remaining Charging Time"]).reset_index(drop=True))

print("\nSOC model dataset shape:", soc_model_df.shape)
print("Remaining-time model dataset shape:",time_model_df.shape)
print("\nMissing values in SOC dataset:")
print(soc_model_df.isna().sum())
print("\nMissing values in remaining-time dataset:")
print(time_model_df.isna().sum())
from sklearn.model_selection import GroupShuffleSplit
def create_group_split(df, group_column, random_seed=42):


    splitter_1 = GroupShuffleSplit(
        n_splits=1,
        test_size=0.30,
        random_state=random_seed
    )
    train_idx, temp_idx = next(
        splitter_1.split(
            df,
            groups=df[group_column]))
    train_df = df.iloc[train_idx].copy()
    temp_df = df.iloc[temp_idx].copy()


    splitter_2 = GroupShuffleSplit(
        n_splits=1,
        test_size=0.50,
        random_state=random_seed
    )
    validation_idx, test_idx = next(
        splitter_2.split(
            temp_df,
            groups=temp_df[group_column]))
    validation_df = temp_df.iloc[validation_idx].copy()
    test_df = temp_df.iloc[test_idx].copy()
    return train_df, validation_df, test_df


RANDOM_SEED = 42
(
    soc_train,
    soc_validation,
    soc_test
) = create_group_split(
    soc_model_df,
    "Source_File",
    RANDOM_SEED
)
print("\nSOC dataset split:")
print("Training:", soc_train.shape)
print("Validation:", soc_validation.shape)
print("Testing:", soc_test.shape)
print("\nSOC split percentages:")

total_soc = len(soc_model_df)
print("Training:",round(len(soc_train) / total_soc * 100, 2),"%")
print("Validation:",round(len(soc_validation) / total_soc * 100, 2),"%")
print("Testing:",round(len(soc_test) / total_soc * 100, 2),"%")


SOC model dataset shape: (14148, 9)
Remaining-time model dataset shape: (44650, 10)

Missing values in SOC dataset:
Voltage             0
Current             0
Temperature         0
Elapsed Time        0
Cycle               0
WhAccu              0
SOC                 0
Source_File         0
Charging_Session    0
dtype: int64

Missing values in remaining-time dataset:
Voltage                        0
Current                        0
Temperature                    0
Elapsed Time                   0
Cycle                          0
Capacity                   30502
WhAccu                     30368
Remaining Charging Time        0
Source_File                    0
Charging_Session               0
dtype: int64

SOC dataset split:
Training: (5645, 9)
Validation: (7376, 9)
Testing: (1127, 9)

SOC split percentages:
Training: 39.9 %
Validation: 52.13 %
Testing: 7.97 %


In [24]:
(
    time_train,
    time_validation,
    time_test
) = create_group_split(
    time_model_df,
    "Source_File",
    RANDOM_SEED
)
def check_group_leakage(
    train_data,
    validation_data,
    test_data
):
    train_groups = set(train_data["Source_File"])
    validation_groups = set(validation_data["Source_File"])
    test_groups = set(test_data["Source_File"])
    train_val_overlap = (train_groups.intersection(validation_groups))
    train_test_overlap = (train_groups.intersection(test_groups))
    validation_test_overlap = (validation_groups.intersection(test_groups))
    print("Train ∩ Validation:",len(train_val_overlap))
    print("Train ∩ Test:",len(train_test_overlap))
    print("Validation ∩ Test:",len(validation_test_overlap))
    if (
        len(train_val_overlap) == 0
        and
        len(train_test_overlap) == 0
        and
        len(validation_test_overlap) == 0
    ):
        print("PASS — No group leakage detected.")
    else:
        raise ValueError("Group leakage detected.")

print("SOC GROUP LEAKAGE CHECK")
check_group_leakage(soc_train,soc_validation,soc_test)

print("REMAINING-TIME GROUP LEAKAGE CHECK")
check_group_leakage(time_train,time_validation,time_test)

SOC GROUP LEAKAGE CHECK
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0
PASS — No group leakage detected.
REMAINING-TIME GROUP LEAKAGE CHECK
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0
PASS — No group leakage detected.


In [25]:
X_soc_train = soc_train[soc_input_variables].copy()
X_soc_validation = soc_validation[soc_input_variables].copy()
X_soc_test = soc_test[soc_input_variables].copy()
y_soc_train = soc_train[["SOC"]].copy()
y_soc_validation = soc_validation[["SOC"]].copy()
y_soc_test = soc_test[["SOC"]].copy()

In [26]:
X_time_train = time_train[time_input_variables].copy()
X_time_validation = time_validation[time_input_variables].copy()
X_time_test = time_test[time_input_variables].copy()
y_time_train = time_train[["Remaining Charging Time"]].copy()
y_time_validation = time_validation[["Remaining Charging Time"]].copy()
y_time_test = time_test[["Remaining Charging Time"]].copy()
soc_preprocessing_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
 )
        ),
        (
            "scaler",
            StandardScaler()
        )])



soc_preprocessing_pipeline.fit(X_soc_train)
X_soc_train_processed = (soc_preprocessing_pipeline.transform(X_soc_train))
X_soc_validation_processed = (soc_preprocessing_pipeline.transform(X_soc_validation))
X_soc_test_processed = (soc_preprocessing_pipeline.transform(X_soc_test))

In [27]:
time_preprocessing_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )])

time_preprocessing_pipeline.fit( X_time_train)
X_time_train_processed = (time_preprocessing_pipeline.transform(X_time_train))
X_time_validation_processed = (time_preprocessing_pipeline.transform(X_time_validation))
X_time_test_processed = (time_preprocessing_pipeline.transform(X_time_test))
print("\nPreprocessing completed.")
print("Imputation and scaling were fitted only on training data.")



Preprocessing completed.
Imputation and scaling were fitted only on training data.


In [28]:
output_directory = ("/content/drive/MyDrive/BATTERY_SOC_PROJECT/day11_outputs")
os.makedirs(
    output_directory,
    exist_ok=True
)
print("\nOutput Directory ")
print(output_directory)
cleaned_dataset_path = os.path.join(output_directory,"cleaned_battery_dataset_day11.csv")
charging_df.to_csv(cleaned_dataset_path,index=False)
print("\nCleaned charging dataset saved:")
print(cleaned_dataset_path)


Output Directory 
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day11_outputs

Cleaned charging dataset saved:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day11_outputs/cleaned_battery_dataset_day11.csv


In [29]:
soc_train.to_csv(os.path.join(output_directory,"SOC_train_day11.csv"),index=False)
soc_validation.to_csv(os.path.join(output_directory,"SOC_validation_day11.csv"),index=False)
soc_test.to_csv(os.path.join(output_directory,"SOC_test_day11.csv"),index=False)
time_train.to_csv(os.path.join(output_directory,"remaining_time_train_day11.csv"),index=False)
time_validation.to_csv(os.path.join(output_directory,"remaining_time_validation_day11.csv"),index=False)
time_test.to_csv(os.path.join(output_directory,"remaining_time_test_day11.csv"),index=False)
joblib.dump(
    soc_preprocessing_pipeline,
    os.path.join(
        output_directory,
        "SOC_preprocessing_pipeline_day11.joblib" ))
joblib.dump(
    time_preprocessing_pipeline,
    os.path.join(
        output_directory,
        "remaining_time_preprocessing_pipeline_day11.joblib"))

['/content/drive/MyDrive/BATTERY_SOC_PROJECT/day11_outputs/remaining_time_preprocessing_pipeline_day11.joblib']

In [30]:
metadata = {
    "random_seed":RANDOM_SEED,
    "SOC_input_variables":soc_input_variables,
    "Remaining_Time_input_variables":time_input_variables,
    "target_variables":target_variables,
    "group_variable":"Source_File",
    "SOC_target_generation":"Capacity progression relative to final ""charging-session capacity",
  "Remaining_Time_target_generation":"Final charging-session timestamp minus ""current timestamp, measured in seconds",
    "feature_selection":"Reserved for Day 13",
    "feature_engineering":"Reserved for Day 13"}
joblib.dump(
    metadata,
    os.path.join(
        output_directory,
        "day11_preprocessing_metadata.joblib"))

['/content/drive/MyDrive/BATTERY_SOC_PROJECT/day11_outputs/day11_preprocessing_metadata.joblib']

In [31]:
print("COMPLETED SUCCESSFULLY")


print("\nTARGET VARIABLES:")
print("1. SOC")
print("2. Remaining Charging Time")
print("\nSOC INPUT VARIABLES:")
for col in soc_input_variables:
    print("-", col)
print("\nREMAINING-TIME INPUT VARIABLES:")
for col in time_input_variables:
    print("-", col)
print("\nSOC DATA SPLIT:")
print("Training   :", len(soc_train))
print("Validation :", len(soc_validation))
print("Testing    :", len(soc_test))
print("\nREMAINING-TIME DATA SPLIT:")
print("Training   :", len(time_train))
print("Validation :", len(time_validation))
print("Testing    :", len(time_test))
print("\nPREPROCESSING:")
print("- Missing values: median imputation")
print("- Scaling: StandardScaler")
print("- Fitted only on training data")
print("- Fixed random seed:", RANDOM_SEED)
print("- Group-based leakage check: PASSED")
print("\nOUTPUT DIRECTORY:")
print(output_directory)
print("\nFeature engineering and final feature selection are reserved for DAY 13.")

COMPLETED SUCCESSFULLY

TARGET VARIABLES:
1. SOC
2. Remaining Charging Time

SOC INPUT VARIABLES:
- Voltage
- Current
- Temperature
- Elapsed Time
- Cycle
- WhAccu

REMAINING-TIME INPUT VARIABLES:
- Voltage
- Current
- Temperature
- Elapsed Time
- Cycle
- Capacity
- WhAccu

SOC DATA SPLIT:
Training   : 5645
Validation : 7376
Testing    : 1127

REMAINING-TIME DATA SPLIT:
Training   : 41015
Validation : 1349
Testing    : 2286

PREPROCESSING:
- Missing values: median imputation
- Scaling: StandardScaler
- Fitted only on training data
- Fixed random seed: 42
- Group-based leakage check: PASSED

OUTPUT DIRECTORY:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day11_outputs

Feature engineering and final feature selection are reserved for DAY 13.


In [32]:
import os

project_path = "/content/drive/MyDrive/BATTERY_SOC_PROJECT"

day11_output_directory = os.path.join(
    project_path,
    "day11_outputs"
)

print("Day 11 output directory:")
print(day11_output_directory)

print("\nFolder exists:",
      os.path.exists(day11_output_directory))

Day 11 output directory:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day11_outputs

Folder exists: True


In [33]:
cleaned_dataset_path = os.path.join(
    day11_output_directory,
    "cleaned_battery_dataset_day11.csv"
)
battery_df.to_csv(cleaned_dataset_path, index=False)
print("CLEANED DATASET SAVED")
print(cleaned_dataset_path)
print("\nShape of cleaned dataset:")
print(battery_df.shape)
print("\nFile exists:",
      os.path.exists(cleaned_dataset_path))

CLEANED DATASET SAVED
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day11_outputs/cleaned_battery_dataset_day11.csv

Shape of cleaned dataset:
(4949249, 19)

File exists: True


In [34]:
import os
import json

day11_output_directory = ("/content/drive/MyDrive/""BATTERY_SOC_PROJECT/day11_outputs")
os.makedirs(day11_output_directory, exist_ok=True)
print("\nDay 11 output folder:")
print(day11_output_directory)

required_files = [
    "cleaned_battery_dataset_day11.csv",
    "SOC_train_day11.csv",
    "SOC_validation_day11.csv",
    "SOC_test_day11.csv",
    "remaining_time_train_day11.csv",
    "remaining_time_validation_day11.csv",
    "remaining_time_test_day11.csv",
    "SOC_preprocessing_pipeline_day11.joblib",
    "remaining_time_preprocessing_pipeline_day11.joblib"
]

print("\nChecking required Day 11 files\n")
missing_files = []
available_files = []
for file_name in required_files:
    file_path = os.path.join(day11_output_directory,file_name)
    if os.path.exists(file_path):
        available_files.append(file_name)
        print("present", file_name)
    else:
        missing_files.append(file_name)
        print("MISSING:", file_name)

metadata_path = os.path.join(day11_output_directory, "metadata_day11.json")
metadata = {
    "project": "Battery SOC Estimation and Remaining Charging Time Prediction using ML",
    "day": "Day 11",
    "random_seed": 42,
    "group_variable": "Source_File",
    "targets": ["SOC","Remaining Charging Time"],
    "soc_input_variables": ["Voltage", "Current","Temperature","Elapsed Time","Cycle"],
    "time_input_variables": ["Voltage", "Current","Temperature","Elapsed Time","Cycle","Capacity"],
    "soc_target_generation": ("SOC was calculated as Capacity divided by the ""final charging capacity of the corresponding charging ""session and expressed as a percentage."),
    "remaining_time_target_generation": (
        "Remaining Charging Time was calculated as the difference "
        "between the final charging timestamp of the session and "
        "the current timestamp." ),
    "data_split": (
        "Approximately 70/15/15 train/validation/test split "
        "using Source_File groups to reduce data leakage."),
    "preprocessing": (
        "Median imputation and StandardScaler were fitted only "
        "on training data and then applied to validation and test data."
    ),
    "feature_selection": "Reserved for Day 13",
    "feature_engineering": "Reserved for Day 13"}
with open(metadata_path, "w") as file:
    json.dump(
        metadata,
        file,
        indent=4
    )
print("\n metadata_day11.json saved")

if os.path.exists(metadata_path):
    print(" Metadata file confirmed")
else:
    print(" Metadata file could not be created")

if len(missing_files) == 0:
    print("\nAll required files are available for Day 12.")
else:
    print("\nAll required files are not available for Day 12.")

    print("\nThe following files are missing:")

    for file_name in missing_files:
        print("-", file_name)
    print("\nRun the corresponding Day 11 sections that create the missing files, then run again.")


print("\nFiles available:",
      len(available_files),
      "/",
      len(required_files))
print("\nDay 11 → Day 12 connection is file-based.")


Day 11 output folder:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day11_outputs

Checking required Day 11 files

present cleaned_battery_dataset_day11.csv
present SOC_train_day11.csv
present SOC_validation_day11.csv
present SOC_test_day11.csv
present remaining_time_train_day11.csv
present remaining_time_validation_day11.csv
present remaining_time_test_day11.csv
present SOC_preprocessing_pipeline_day11.joblib
present remaining_time_preprocessing_pipeline_day11.joblib

 metadata_day11.json saved
 Metadata file confirmed

All required files are available for Day 12.

Files available: 9 / 9

Day 11 → Day 12 connection is file-based.
